# 🚀 vLLM Setup sur Google Colab

Ce notebook configure vLLM avec GPU NVIDIA pour votre projet RAG.

**⚠️ Important** : Activez le GPU dans Colab : `Runtime > Change runtime type > GPU (T4 ou A100)`


## Étape 1 : Installation des dépendances


In [7]:
# Installer les dépendances avec les bonnes versions
%pip install --upgrade pip setuptools wheel
%pip install -q jedi>=0.16
%pip install -q qdrant-client fastembed
%pip install -q vllm transformers torch sentence-transformers requests
%pip install -q uvicorn
# Corriger les conflits de dépendances
%pip install -q --force-reinstall pydantic
%pip install -q --force-reinstall starlette
%pip install -q --force-reinstall fastapi
print("✅ Installation terminée - vérification des versions...")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.20.0 requires fastapi<0.119.0,>=0.115.0, but you have fastapi 0.124.0 which is incompatible.
gradio 5.50.0 requires pydantic<=2.12.3,>=2.0, but you have pydantic 2.12.5 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.20.0 requires fastapi<0.119.0,>=0.115.0, but you have fastapi 0.124.0 which is incompatible.
gradio 5.50.0 requires pydantic<=2.12.3,>=2.0, but you have pydantic 2.12.5 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.20.0 requires fastapi<0.119.0,>=0.115.0, but you hav

## Étape 2 : Vérifier que le GPU est disponible


In [8]:
# Vérifier les versions des packages critiques
import subprocess
import sys

packages_to_check = ['fastapi', 'starlette', 'pydantic', 'vllm', 'torch']

for package in packages_to_check:
    result = subprocess.run([sys.executable, '-m', 'pip', 'show', package], 
                          capture_output=True, text=True)
    for line in result.stdout.split('\n'):
        if line.startswith('Version:'):
            print(f"✅ {package}: {line.split('Version: ')[1]}")

✅ fastapi: 0.124.0
✅ starlette: 0.50.0
✅ starlette: 0.50.0
✅ pydantic: 2.12.5
✅ pydantic: 2.12.5
✅ vllm: 0.12.0
✅ vllm: 0.12.0
✅ torch: 2.9.0+cu126
✅ torch: 2.9.0+cu126


In [9]:
import torch

if torch.cuda.is_available():
    print(f"✅ GPU disponible : {torch.cuda.get_device_name(0)}")
    print(f"✅ Mémoire GPU : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("❌ Pas de GPU détecté! Activez le GPU dans Runtime > Change runtime type")
    raise RuntimeError("GPU requis pour vLLM")


❌ Pas de GPU détecté! Activez le GPU dans Runtime > Change runtime type


RuntimeError: GPU requis pour vLLM

## Étape 3 : Lancer vLLM en arrière-plan


In [ ]:
import subprocess
import threading
import time
import requests
import os

def start_vllm_server():
    """Lance le serveur vLLM avec API OpenAI-compatible."""
    os.system("python -m vllm.entrypoints.openai.api_server \
              --model TheBloke/Mistral-7B-Instruct-v0.2-AWQ \
              --quantization awq \
              --dtype float16 \
              --port 8000 \
              --host 0.0.0.0 \
              > /dev/null 2>&1")

# Lancer vLLM en arrière-plan
print("⏳ Démarrage de vLLM (peut prendre 2-3 minutes pour télécharger le modèle)...")
thread = threading.Thread(target=start_vllm_server, daemon=True)
thread.start()

# Attendre que le serveur démarre
max_wait = 300  # 5 minutes max
wait_time = 0
while wait_time < max_wait:
    try:
        resp = requests.get("http://localhost:8000/health", timeout=2)
        if resp.status_code == 200:
            print("✅ vLLM est prêt et accessible sur http://localhost:8000")
            break
    except:
        pass
    time.sleep(5)
    wait_time += 5
    if wait_time % 30 == 0:
        print(f"⏳ Encore en attente... ({wait_time}s / {max_wait}s)")

if wait_time >= max_wait:
    print("❌ Timeout: vLLM n'a pas démarré dans les temps")
    print("💡 Vérifiez les logs ou réessayez avec un modèle plus petit")


## Étape 4 : Tester l'API vLLM


In [ ]:
import requests
import json

# Test de l'API
response = requests.post(
    "http://localhost:8000/v1/chat/completions",
    json={
        "model": "TheBloke/Mistral-7B-Instruct-v0.2-AWQ",
        "messages": [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": "What is the capital of France? Answer in one sentence."}
        ],
        "temperature": 0.7,
        "max_tokens": 100
    },
    timeout=30
)

if response.status_code == 200:
    result = response.json()
    print("✅ API fonctionne!")
    print(f"📝 Réponse : {result['choices'][0]['message']['content']}")
    print(f"⏱️  Tokens utilisés : {result['usage']['total_tokens']}")
else:
    print(f"❌ Erreur : {response.status_code}")
    print(response.text)


## Étape 5 : Exposer l'API publiquement (optionnel - pour tester depuis votre machine locale)


In [ ]:
# Installer ngrok pour exposer le port 8000
%pip install -q pyngrok

from pyngrok import ngrok

# Exposer le port 8000
public_url = ngrok.connect(8000)
print(f"🌐 API publique accessible via : {public_url}")
print(f"\n📋 Utilisez cette URL dans votre .env local :")
print(f"LLM_BASE_URL={public_url}/v1")
print(f"LLM_API_KEY=dummy-key")
print(f"\n⚠️  Cette URL est publique et temporaire (expire quand Colab se ferme)")


# 🚀 vLLM Setup sur Google Colab

Ce notebook lance vLLM avec GPU pour votre projet RAG.

**Important** : Activez le GPU dans Runtime → Change runtime type → GPU (T4)
